# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [ ]:
# Write your code below.
%load_ext dotenv
%dotenv 

#test test


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from glob import glob

# Write your code below.

price_data_dir = os.getenv("PRICE_DATA")       # absolute or relative folder that holds Parquet files
if price_data_dir is None:
    raise ValueError("Environment variable PRICE_DATA is not set. Check your .env file.")

price_data_dir




'../../05_src/data/prices/'

In [ ]:
#price_data_files=glob(os.path.join(price_data_dir,"*.parquet"))

price_data_files = glob(
    os.path.join(price_data_dir, "**", "*.parquet"),
    recursive=True,
)

#i fixed one error here - instead of only using "*.parquet", i added "**", beasue these parquet files might be one level down

In [27]:
len(price_data_files)

2010

In [25]:
#this is not required by the assignment but I want to double check if these parquet files were created successfully
import pyarrow.parquet as pq
from pprint import pprint

px_dd = dd.read_parquet(price_data_files)

In [26]:
#this entire section is for my error fixing, initially i only go one level down to find these parquet files so this block of codes to validate whether
#there were any parquet files being passed

print("PRICE_DATA =", os.getenv("PRICE_DATA"))
print("price_data_files →", len(price_data_files), "files")
for p in price_data_files[:5]:
    print("  •", p)


PRICE_DATA = ../../05_src/data/prices/
price_data_files → 2010 files
  • ../../05_src/data/prices/BKTI/BKTI_2012/part.0.parquet
  • ../../05_src/data/prices/BKTI/BKTI_2012/part.1.parquet
  • ../../05_src/data/prices/BKTI/BKTI_2015/part.0.parquet
  • ../../05_src/data/prices/BKTI/BKTI_2015/part.1.parquet
  • ../../05_src/data/prices/BKTI/BKTI_2014/part.0.parquet


In [30]:
#import dask
import dask.dataframe as dd
px_dd = dd.read_parquet(price_data_files)


In [36]:
"ticker" in px_dd.columns

#i want to see if "ticker" is a column inside of the px_dd, which is a dask dataframe, then i can use little compute power to get a list



True

In [38]:
print(px_dd.columns.tolist())

['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'ticker', 'Year']


In [39]:
print(px_dd.head()) 

Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj Close, Volume, source, ticker, Year]
Index: []


/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/dask/dataframe/core.py:8153: UserWarning: Insufficient elements for `head`. 5 elements requested, only 0 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(


/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/dask/dataframe/core.py:8153: UserWarning: Insufficient elements for `head`. 5 elements requested, only 0 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(
/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/dask/dataframe/core.py:8153: UserWarning: Insufficient elements for `head`. 5 elements requested, only 0 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(
/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/dask/dataframe/core.py:8153: UserWarning: Insufficient elements for `head`. 5 elements requested, only 0 elements available. Try passing larger `npartitions` to `head`.
  warnings.warn(
/opt/miniconda3/envs/dsi_participant/lib/python3.9/site-packages/dask/dataframe/core.py:8153: UserWarning: Insufficient elements for `head`. 5 elements requested, only 0 elements available. Try passing larger `npartitions` to `head`.
  warnings.wa

In [40]:
print("Files found:", len(price_data_files))
print("Columns:", px_dd.columns.tolist())          # should list column names

# Row count (lazy until compute)
print("Row estimate:", px_dd.size.compute())       # or px_dd.shape[0].compute()


Files found: 2010
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source', 'ticker', 'Year']
Row estimate: 2302730


In [41]:
#because i have empty dataframe after print px_dd() but after I run the prior code to confirm these are not empty. i want to test a few more

# Ask Dask to keep scanning partitions until it collects 5 rows
px_dd.head(5, npartitions=-1)

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year
86576,2015-01-02,26.219999,26.379999,25.770000,26.160000,26.160000,547100.0,LYV.csv,LYV,2015
86577,2015-01-05,25.959999,26.080000,25.530001,25.559999,25.559999,615000.0,LYV.csv,LYV,2015
86578,2015-01-06,25.570000,25.570000,24.709999,24.820000,24.820000,982700.0,LYV.csv,LYV,2015
86579,2015-01-07,25.010000,25.160000,24.770000,25.110001,25.110001,828700.0,LYV.csv,LYV,2015
86580,2015-01-08,25.389999,25.940001,25.320000,25.830000,25.830000,870800.0,LYV.csv,LYV,2015


In [33]:
px_dd = px_dd.shuffle(on="ticker", shuffle="tasks")

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.

#at this point, I have validated the panquet data have been created and passed to datafram called px_dd, now it is time to add additional data columns as needed
#returns: (close / close_lag_1) - 1, with the formula, we have close column but we need to derive close_lag_1, which is the previous data close price for the right ticket


dd_shift = px_dd.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)
#use apply function because i need to create a new column



In [ ]:
# 0. (Optional but recommended) move each ticker to exactly one partition
px_dd = px_dd.shuffle(on="ticker", shuffle="tasks")



In [43]:
# 1. helper applied to each pandas partition (one ticker per partition)
def _add_close_lag(pdf):
    pdf = pdf.sort_values("Date")                # ensure chronological
    pdf["Close_lag_1"] = pdf["Close"].shift(1)
    return pdf

In [45]:
import numpy as np

# 2. tell Dask what new columns will look like
meta = (
    px_dd._meta
         .assign(Close_lag_1=np.float64())
)

In [46]:
# 3. groupby-apply with meta
px_dd = (
    px_dd.groupby("ticker", dropna=False)
         .apply(_add_close_lag, meta=meta)
)

# quick peek
print(px_dd.head())

Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj Close, Volume, source, ticker, Year, Close_lag_1]
Index: []


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.

px_dd.head(5 , npartitions= -1)

#this validate all operations are correct, we have calcualted close_lag_1

Date   Open   High    Low  Close  Adj Close     Volume  \
ticker                                                                      
LYV    84304 2005-12-21  11.00  11.05  10.55  10.85      10.85  5796900.0   
       84305 2005-12-22  10.60  11.12  10.59  10.60      10.60  4152500.0   
       84306 2005-12-23  10.90  12.78  10.90  11.95      11.95  8844700.0   
       84307 2005-12-27  12.20  12.79  12.13  12.50      12.50  3324100.0   
       84308 2005-12-28  12.25  14.00  12.25  13.30      13.30  3187700.0   

               source ticker  Year  Close_lag_1  
ticker                                           
LYV    84304  LYV.csv    LYV  2005          NaN  
       84305  LYV.csv    LYV  2005        10.85  
       84306  LYV.csv    LYV  2005        10.60  
       84307  LYV.csv    LYV  2005        11.95  
       84308  LYV.csv    LYV  2005        12.50

In [50]:
#let's continue with return using the exact same function

px_dd =px_dd.assign (

    returns = (px_dd['Close'] - px_dd['Close_lag_1'])/px_dd['Close_lag_1']
)

In [51]:
px_dd.head(5 , npartitions= -1)

Date   Open   High    Low  Close  Adj Close     Volume  \
ticker                                                                      
LYV    84304 2005-12-21  11.00  11.05  10.55  10.85      10.85  5796900.0   
       84305 2005-12-22  10.60  11.12  10.59  10.60      10.60  4152500.0   
       84306 2005-12-23  10.90  12.78  10.90  11.95      11.95  8844700.0   
       84307 2005-12-27  12.20  12.79  12.13  12.50      12.50  3324100.0   
       84308 2005-12-28  12.25  14.00  12.25  13.30      13.30  3187700.0   

               source ticker  Year  Close_lag_1   returns  
ticker                                                     
LYV    84304  LYV.csv    LYV  2005          NaN       NaN  
       84305  LYV.csv    LYV  2005        10.85 -0.023041  
       84306  LYV.csv    LYV  2005        10.60  0.127358  
       84307  LYV.csv    LYV  2005        11.95  0.046025  
       84308  LYV.csv    LYV  2005        12.50  0.064000

In [52]:
px_dd = px_dd.assign (

    hi_lo_range = px_dd['High'] - px_dd['Low']
)


In [53]:
px_dd.head(5 , npartitions= -1)

Date   Open   High    Low  Close  Adj Close     Volume  \
ticker                                                                      
LYV    84304 2005-12-21  11.00  11.05  10.55  10.85      10.85  5796900.0   
       84305 2005-12-22  10.60  11.12  10.59  10.60      10.60  4152500.0   
       84306 2005-12-23  10.90  12.78  10.90  11.95      11.95  8844700.0   
       84307 2005-12-27  12.20  12.79  12.13  12.50      12.50  3324100.0   
       84308 2005-12-28  12.25  14.00  12.25  13.30      13.30  3187700.0   

               source ticker  Year  Close_lag_1   returns  hi_lo_range  
ticker                                                                  
LYV    84304  LYV.csv    LYV  2005          NaN       NaN         0.50  
       84305  LYV.csv    LYV  2005        10.85 -0.023041         0.53  
       84306  LYV.csv    LYV  2005        10.60  0.127358         1.88  
       84307  LYV.csv    LYV  2005        11.95  0.046025         0.66  
       84308  LYV.csv    LYV  2005        12.50  0.064000         1.75

In [66]:
px_dd.head()

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,returns,hi_lo_range


In [ ]:
dd_feat = px_dd.persist()
print(dd_feat.head())

#.persist() eeps data in Dask format, but materialized in memory; you can chain more lazy Dask ops quickly.

Empty DataFrame
Columns: [Date, Open, High, Low, Close, Adj Close, Volume, source, ticker, Year, Close_lag_1, returns, hi_lo_range]
Index: []


In [69]:
pdf = dd_feat.compute()

In [72]:
print(pdf.dtypes['Date'])

datetime64[ns]


In [75]:
if 'ticker' in pdf.columns:
    pdf = pdf.drop(columns='ticker')


In [76]:
# 1. Reset all index levels into columns (drops them from the index)
pdf = pdf.reset_index()  

In [77]:
pdf = pdf.sort_values(by=['ticker', 'Date'])

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [64]:
pdf.set_index('tickets')

KeyError: "None of ['tickets'] are in the columns"

In [59]:
print(dd_feat.columns)


Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year', 'Close_lag_1', 'returns', 'hi_lo_range'],
      dtype='object')


In [78]:
pdf['returns_ma_10'] = (
    pdf.groupby('ticker')['returns']         # stay inside each symbol
       .transform(lambda s: s.rolling(10, min_periods=1).mean())
)


In [79]:
pdf.describe()

,level_1,Date,Open,High,Low,Close,Adj Close,Volume,Year,Close_lag_1,returns,hi_lo_range,returns_ma_10
count,230273.000000,230273,230260.000000,230260.000000,230260.000000,230260.000000,230260.000000,2.302600e+05,230273.000000,230201.000000,230195.000000,230260.000000,230212.000000
mean,115136.000000,2007-12-23 10:50:18.882803968,60.106993,62.942579,59.870115,61.372954,57.501995,1.828409e+06,2007.476712,61.382137,0.894288,3.072464,0.894414
min,0.000000,1972-06-01 00:00:00,0.000000,0.001000,0.001000,0.001000,0.000925,0.000000e+00,1972.000000,0.001000,-0.999878,0.000000,-0.937587
25%,57568.000000,2002-06-26 00:00:00,4.619983,4.850000,4.650000,4.750000,2.925355,1.120000e+04,2002.000000,4.750000,-0.011236,0.090000,-0.003435
50%,115136.000000,2010-06-21 00:00:00,11.720000,12.150000,11.760000,11.969445,9.120000,1.043000e+05,2010.000000,11.970000,0.000000,0.280000,0.000249
75%,172704.000000,2015-11-30 00:00:00,27.459999,28.530001,27.615850,28.052499,23.523773,8.426250e+05,2015.000000,28.059999,0.010870,0.739998,0.004241
max,230272.000000,2020-04-02 00:00:00,23849.500000,23849.500000,23849.500000,23849.500000,22049.511719,2.452292e+08,2020.000000,23849.500000,149058.378332,2762.500000,14926.618712
std,66474.233606,NaN,505.652448,520.366779,490.585454,505.222947,504.815392,5.808363e+06,10.068990,505.286564,313.852304,35.010470,99.368038


In [ ]:
#No.
#Dask has its own groupby, rolling, and mean APIs, so you can compute a per-ticker rolling average entirely inside Dask:

This is more efficient code then the one I did - i tested one by one but this is all in one group

START with the list of definitions - create a function - all the transformation we need 

import dask.dataframe as dd
import numpy as np
import os
from glob import glob

# ---------------------------------------------------------
# 1.  Read all parquet files into a single Dask DataFrame
# ---------------------------------------------------------
price_data_dir   = os.getenv("PRICE_DATA")                 # directory path
price_data_files = glob(os.path.join(price_data_dir, "**", "*.parquet"),
                        recursive=True)

px_dd = dd.read_parquet(price_data_files)                  # lazy load

# ---------------------------------------------------------
# 2.  Make sure each ticker sits in exactly one partition
#     (so .shift(1) sees yesterday's row)
# ---------------------------------------------------------
px_dd = px_dd.shuffle(on="ticker", shuffle="tasks")

# ---------------------------------------------------------
# 3.  Helper to add the four features for ONE ticker
#     (runs on a pandas DataFrame inside each partition)
# ---------------------------------------------------------
def _add_features(pdf):
    pdf = pdf.sort_values("Date")                          # chronological
    pdf["Close_lag_1"]      = pdf["Close"].shift(1)
    pdf["Adj_Close_lag_1"]  = pdf["Adj_Close"].shift(1)
    pdf["returns"]          = pdf["Close"] / pdf["Close_lag_1"] - 1
    pdf["hi_lo_range"]      = pdf["High"]  - pdf["Low"]
    return pdf

# ---------------------------------------------------------
# 4.  meta tells Dask the schema after the transformation
# ---------------------------------------------------------
meta = (
    px_dd._meta
         .assign(
             Close_lag_1     = np.float64(),
             Adj_Close_lag_1 = np.float64(),
             returns         = np.float64(),
             hi_lo_range     = np.float64()
         )
)

# ---------------------------------------------------------
# 5.  Apply the helper per ticker and get the final DF
# ---------------------------------------------------------
dd_feat = (
    px_dd.groupby("ticker", dropna=False)
         .apply(_add_features, meta=meta)
         .persist()                      # optional: keep partitions in RAM
)

# ---------------------------------------------------------
# 6.  Quick sanity peek (tiny compute)
# ---------------------------------------------------------
print(dd_feat.head())
print(dd_feat.columns)


## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.